# Stage 02 - Model HPO Train-Inner

Route: `lst_models` V2
Scope: `validation_only`
User-facing notebook: `notebooks/02_model_hpo_train_inner_colab.ipynb`

This notebook runs the Stage 02 package entry point against the exact Stage 01 handoff. Stage 02 is train-inner HPO only. It does not read official validation or closed holdout/test rows and does not determine the final model.


## Protocol Summary

Stage 02 consumes `01_candidate_inputs.json` from one frozen Stage 01 run folder. Stage 02 is train-inner HPO only: it does not read, rank, tune against, or summarize official validation, test, or holdout rows.

The current executable layer is an HPO planning scaffold. It builds planned rows from Stage 01 candidate inputs, approved families, search profiles, train-inner folds, and seeds. It marks those rows as `skipped_not_implemented`, writes no frozen params, and keeps `ready_for_stage03=false`.

Stage 02 must run all Stage 01-approved families that are enabled in the config, or block if the budget does not close. It must not secretly drop or demote approved families. Current enabled HPO families:

- `lightgbm`
- `standard_dlinear`
- `tcn`
- `ms_dlinear_tcn`

`simple_gru` and `shallow_lstm` remain optional fixed controls, not core HPO families. Stage 00 baseline registry names remain `stratified_dummy_train_prior`, `majority_train_prior`, `constant_up`, and `constant_down`.


## Expected Artifacts

Stage 02 writes a compact run folder when `RUN_STAGE02=True`:

```text
results/02_model_hpo_train_inner/<run_id>/
  run_manifest.json
  artifact_inventory.csv
  02_model_hpo_train_inner_summary.csv
  02_hpo_plan_ledger.csv
  02_best_params_by_family.json
  02_stage03_handoff.json
```

The current package implementation is deliberately honest: it blocks when Stage 01 has no candidates and does not fabricate HPO metrics.

Formal HPO outputs such as `02_hpo_trial_ledger.csv`, `02_hpo_summary.csv`, `02_baseline_control_summary.csv`, and `02_frozen_candidate.json` are future-layer artifacts. They are not active until config, code, notebook, and tests are updated together.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib
import json
import shutil
import subprocess
import sys
import zipfile

RUN_PROJECT_BOOTSTRAP = True
PROJECT_BOOTSTRAP_MODE = "github_commit"  # github_commit | drive_bundle | manual_upload | already_present

PROJECT_REPO_URL = "https://github.com/zkc768/lstm-zhang.git"
PROJECT_REPO_COMMIT = "b6a67edde32c619b649ca1dbcae9bf869bee6833"
PROJECT_ROOT = Path("/content/lst_models")
PROJECT_DRIVE_BUNDLE_FILE_ID = ""
PROJECT_DRIVE_BUNDLE_NAME = "lst_models_stage02_colab_project.zip"

RUN_STAGE01_DRIVE_SYNC = True
RUN_STAGE02_CHECKPOINT = False
RUN_STAGE02_DRIVE_BACKUP = True
RUN_STAGE02 = False
RUN_ARTIFACT_DISPLAY = False

STAGE_NAME = "02_model_hpo_train_inner"
ROUTE = "lst_models"
SCOPE = "validation_only"
HOLDOUT_TEST_CONTACT = False
STAGE01_RUN_ID = "20260609_070204"
STAGE01_OUTPUT_DIR = Path("/content/lst_models_results/01_feature_window_search") / STAGE01_RUN_ID
STAGE01_DRIVE_PATH_PARTS = ["lst_models", "results", "01_feature_window_search", STAGE01_RUN_ID]
OUTPUT_DIR = Path("/content/lst_models_results/02_model_hpo_train_inner")
STAGE02_DRIVE_RESULT_PATH_PARTS = ["lst_models", "results", "02_model_hpo_train_inner"]
CHECKPOINT_ROOT = Path("/content/lst_models_checkpoints/02_model_hpo_train_inner")
CHECKPOINT_DRIVE_PATH_PARTS = ["lst_models", "checkpoints", "02_model_hpo_train_inner", STAGE01_RUN_ID]
CORE_HPO_FAMILIES = ["lightgbm", "standard_dlinear", "tcn", "ms_dlinear_tcn"]
BASELINE_REGISTRY_NAMES = ["stratified_dummy_train_prior", "majority_train_prior", "constant_up", "constant_down"]

def quote_drive_query_value(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")

def run_cmd(args, cwd=None):
    print("+", " ".join(str(arg) for arg in args))
    subprocess.run(args, cwd=cwd, check=True)

def looks_like_project_root(path):
    return (
        (path / "configs" / "stages" / "02_model_hpo_train_inner.yaml").exists()
        and (path / "docs" / "protocols" / "02_model_hpo_train_inner_protocol.md").exists()
        and (path / "notebooks" / "02_model_hpo_train_inner_colab.ipynb").exists()
        and (path / "src" / "lst_models" / "stages" / "model_hpo_train_inner.py").exists()
    )

def safe_extract_project_zip(zip_path):
    destination = Path("/content").resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
            target = (destination / member_path).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
        archive.extractall(destination)
    candidates = [Path("/content/lst_models"), Path("/content") / zip_path.stem]
    for candidate in candidates:
        if looks_like_project_root(candidate):
            return candidate
    raise FileNotFoundError(
        "The project bundle was extracted, but no lst_models project root was found. "
        "The bundle must contain configs/, docs/, notebooks/, and src/."
    )

def download_and_extract_project_zip_from_drive():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
    except ImportError as exc:
        raise RuntimeError(
            "PROJECT_BOOTSTRAP_MODE='drive_bundle' only works inside Colab "
            "with Google Drive API dependencies available."
        ) from exc
    file_id = PROJECT_DRIVE_BUNDLE_FILE_ID.strip()
    if not file_id:
        raise ValueError("PROJECT_DRIVE_BUNDLE_FILE_ID is required for drive_bundle mode.")
    auth.authenticate_user()
    service = build("drive", "v3")
    zip_path = Path("/content") / PROJECT_DRIVE_BUNDLE_NAME
    request = service.files().get_media(fileId=file_id)
    with zip_path.open("wb") as file_handle:
        downloader = MediaIoBaseDownload(file_handle, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return safe_extract_project_zip(zip_path)

def upload_and_extract_project_zip():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("PROJECT_BOOTSTRAP_MODE='manual_upload' only works inside Colab.") from exc
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No project zip was uploaded.")
    zip_names = [name for name in uploaded if name.endswith(".zip")]
    if not zip_names:
        raise ValueError("Upload a .zip file containing the lst_models project folder.")
    return safe_extract_project_zip(Path("/content") / zip_names[0])

if RUN_PROJECT_BOOTSTRAP:
    if PROJECT_BOOTSTRAP_MODE == "github_commit":
        if (PROJECT_ROOT / ".git").exists():
            run_cmd(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        else:
            run_cmd(["git", "clone", PROJECT_REPO_URL, str(PROJECT_ROOT)])
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
        assert actual_commit == PROJECT_REPO_COMMIT, (actual_commit, PROJECT_REPO_COMMIT)
    elif PROJECT_BOOTSTRAP_MODE == "drive_bundle":
        PROJECT_ROOT = download_and_extract_project_zip_from_drive()
    elif PROJECT_BOOTSTRAP_MODE == "manual_upload":
        PROJECT_ROOT = upload_and_extract_project_zip()
    elif PROJECT_BOOTSTRAP_MODE == "already_present":
        pass
    else:
        raise ValueError("PROJECT_BOOTSTRAP_MODE must be github_commit, drive_bundle, manual_upload, or already_present")

STAGE_CONFIG_PATH = PROJECT_ROOT / "configs" / "stages" / "02_model_hpo_train_inner.yaml"
PROTOCOL_PATH = PROJECT_ROOT / "docs" / "protocols" / "02_model_hpo_train_inner_protocol.md"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "02_model_hpo_train_inner_colab.ipynb"
STAGE_ENTRYPOINT_PATH = PROJECT_ROOT / "src" / "lst_models" / "stages" / "model_hpo_train_inner.py"
SEARCH_SPACE_PATHS = [
    PROJECT_ROOT / "configs" / "models" / family / "search_space.yaml"
    for family in CORE_HPO_FAMILIES
]
REQUIRED_PROJECT_FILES = [STAGE_CONFIG_PATH, PROTOCOL_PATH, NOTEBOOK_PATH, STAGE_ENTRYPOINT_PATH, *SEARCH_SPACE_PATHS]
missing_project_files = [path for path in REQUIRED_PROJECT_FILES if not path.exists()]
if missing_project_files:
    missing_text = "\n".join(f"- {path}" for path in missing_project_files)
    raise FileNotFoundError(
        "Stage 02 bootstrap target is missing required package sidecars. "
        "For normal Colab use, push the full stage bundle to GitHub and update "
        "PROJECT_REPO_COMMIT to that exact commit. Missing files:\n" + missing_text
    )

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

def clear_project_import_cache():
    cached = [name for name in sys.modules if name == "lst_models" or name.startswith("lst_models.")]
    for name in cached:
        del sys.modules[name]
    importlib.invalidate_caches()

clear_project_import_cache()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_BOOTSTRAP_MODE:", PROJECT_BOOTSTRAP_MODE)
print("PROJECT_REPO_URL:", PROJECT_REPO_URL)
print("PROJECT_COMMIT:", PROJECT_REPO_COMMIT)
print("SRC_PATH:", SRC_PATH)
print("STAGE_CONFIG_PATH:", STAGE_CONFIG_PATH)
print("PROTOCOL_PATH:", PROTOCOL_PATH)
print("NOTEBOOK_PATH:", NOTEBOOK_PATH)
print("STAGE_ENTRYPOINT_PATH:", STAGE_ENTRYPOINT_PATH)
print("STAGE01_RUN_ID:", STAGE01_RUN_ID)
print("STAGE01_OUTPUT_DIR:", STAGE01_OUTPUT_DIR)
print("STAGE01_DRIVE_PATH_PARTS:", STAGE01_DRIVE_PATH_PARTS)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("STAGE02_DRIVE_RESULT_PATH_PARTS:", STAGE02_DRIVE_RESULT_PATH_PARTS)
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("CHECKPOINT_DRIVE_PATH_PARTS:", CHECKPOINT_DRIVE_PATH_PARTS)
print("RUN_STAGE01_DRIVE_SYNC:", RUN_STAGE01_DRIVE_SYNC)
print("RUN_STAGE02_CHECKPOINT:", RUN_STAGE02_CHECKPOINT)
print("RUN_STAGE02_DRIVE_BACKUP:", RUN_STAGE02_DRIVE_BACKUP)
print("RUN_STAGE02:", RUN_STAGE02)
print("RUN_ARTIFACT_DISPLAY:", RUN_ARTIFACT_DISPLAY)


## Config Load And Contract Check

This cell reads the Stage 02 config sidecar and checks the notebook-facing contract. It does not fit models or read official validation metrics.


In [ ]:
try:
    import yaml
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "PyYAML is required to read the Stage 02 config. Install project dependencies before running."
    ) from exc

def require_path(path):
    if not path.exists():
        raise FileNotFoundError(f"missing required Stage 02 file: {path}")

require_path(STAGE_CONFIG_PATH)
require_path(PROTOCOL_PATH)

with STAGE_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    stage02_config = yaml.safe_load(handle)

stage01_inputs = stage02_config["inputs"]
assert stage02_config["stage_name"] == STAGE_NAME
assert stage02_config["route"] == ROUTE
assert stage02_config["scope"] == SCOPE
assert stage02_config["holdout_test_contact"] is HOLDOUT_TEST_CONTACT
assert stage01_inputs["stage01_run_id"] == STAGE01_RUN_ID
assert Path(stage01_inputs["stage01_runtime_run_dir"]) == STAGE01_OUTPUT_DIR
assert stage01_inputs["stage01_drive_path_parts"] == STAGE01_DRIVE_PATH_PARTS
assert [family for family, spec in stage02_config["hpo_families"].items() if spec["enabled"]] == CORE_HPO_FAMILIES
assert stage02_config["selection_rules"]["no_official_validation_selection"] is True
assert stage02_config["selection_rules"]["no_final_model_selected"] is True

profile_counts = {}
for family in CORE_HPO_FAMILIES:
    search_space_path = PROJECT_ROOT / stage02_config["hpo_families"][family]["search_space"]
    require_path(search_space_path)
    with search_space_path.open("r", encoding="utf-8") as handle:
        search_space = yaml.safe_load(handle)
    profile_counts[family] = len(search_space.get("profiles", []))
per_candidate_input_plan_rows = (
    sum(profile_counts.values())
    * int(stage02_config["train_inner"]["n_folds"])
    * len(stage02_config["train_inner"]["seeds"])
)
max_candidate_inputs_under_cap = stage02_config["budget"]["max_hpo_plan_rows"] // per_candidate_input_plan_rows

print(json.dumps({
    "stage_name": stage02_config["stage_name"],
    "scope": stage02_config["scope"],
    "source_stage01_run_id": stage01_inputs["stage01_run_id"],
    "stage01_runtime_run_dir": stage01_inputs["stage01_runtime_run_dir"],
    "stage01_drive_path_parts": stage01_inputs["stage01_drive_path_parts"],
    "core_hpo_families": CORE_HPO_FAMILIES,
    "baseline_registry_names": BASELINE_REGISTRY_NAMES,
    "profile_counts": profile_counts,
    "per_candidate_input_plan_rows": per_candidate_input_plan_rows,
    "max_candidate_inputs_under_cap": max_candidate_inputs_under_cap,
    "max_hpo_plan_rows": stage02_config["budget"]["max_hpo_plan_rows"],
    "holdout_test_contact": stage02_config["holdout_test_contact"],
}, indent=2))


## Stage 01 Input Check

Stage 02 requires frozen Stage 01 artifacts from one exact run folder. When `RUN_STAGE02=True` or `RUN_STAGE02_CHECKPOINT=True`, this cell first checks the runtime path. If files are missing and `RUN_STAGE01_DRIVE_SYNC=True`, it downloads the exact Stage 01 run folder files from Drive path parts in the config.


In [ ]:
from lst_models.artifacts import require_artifacts

def find_unique_drive_child(service, parent_id, name, mime_type=None):
    escaped_name = quote_drive_query_value(name)
    query_parts = [f"name = '{escaped_name}'", f"'{parent_id}' in parents", "trashed = false"]
    if mime_type:
        query_parts.append(f"mimeType = '{mime_type}'")
    response = service.files().list(
        q=" and ".join(query_parts),
        spaces="drive",
        fields="files(id, name, mimeType, size)",
        pageSize=10,
    ).execute()
    matches = response.get("files", [])
    if len(matches) != 1:
        raise FileNotFoundError(
            f"expected exactly one Drive item named {name!r} under parent {parent_id}; found {len(matches)}"
        )
    return matches[0]

def resolve_drive_folder(service, path_parts):
    folder_id = "root"
    folder_mime = "application/vnd.google-apps.folder"
    for folder_name in path_parts:
        folder = find_unique_drive_child(service, folder_id, folder_name, folder_mime)
        folder_id = folder["id"]
    return folder_id

def download_drive_file(service, file_id, output_path):
    from googleapiclient.http import MediaIoBaseDownload
    output_path.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id)
    with output_path.open("wb") as handle:
        downloader = MediaIoBaseDownload(handle, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()

def sync_stage01_artifacts_from_drive(required_names):
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError(
            "Stage 01 Drive sync only works inside Colab with Google API dependencies. "
            f"Alternatively, place the frozen Stage 01 run folder at {STAGE01_OUTPUT_DIR}."
        ) from exc
    auth.authenticate_user()
    service = build("drive", "v3")
    run_folder_id = resolve_drive_folder(service, STAGE01_DRIVE_PATH_PARTS)
    for artifact_name in required_names:
        output_path = STAGE01_OUTPUT_DIR / artifact_name
        if output_path.exists():
            continue
        metadata = find_unique_drive_child(service, run_folder_id, artifact_name)
        download_drive_file(service, metadata["id"], output_path)
        print(f"Downloaded Stage 01 artifact: {output_path}")

required_stage01_artifacts = stage02_config["inputs"]["required_stage01_artifacts"]
if RUN_STAGE02 or RUN_STAGE02_CHECKPOINT:
    missing_before = [name for name in required_stage01_artifacts if not (STAGE01_OUTPUT_DIR / name).exists()]
    if missing_before and RUN_STAGE01_DRIVE_SYNC:
        print("Missing Stage 01 artifacts in runtime; syncing exact frozen run from Drive.")
        print("Drive path parts:", STAGE01_DRIVE_PATH_PARTS)
        sync_stage01_artifacts_from_drive(missing_before)
    stage01_paths = require_artifacts(STAGE01_OUTPUT_DIR, required_stage01_artifacts)
    print("Stage 01 artifact presence check passed.")
    print(json.dumps({name: str(path) for name, path in stage01_paths.items()}, indent=2))
else:
    print("RUN_STAGE02=False and RUN_STAGE02_CHECKPOINT=False; Stage 01 artifact check not executed.")


## Optional Drive Checkpoint

Colab runtimes are temporary, so this notebook can write a compact checkpoint archive to Google Drive when `RUN_STAGE02_CHECKPOINT=True`. The checkpoint uses the Drive API and a compressed archive instead of running HPO directly from a mounted Drive folder. This keeps Drive I/O small and preserves enough Stage 02-safe state to reconstruct the run after a Colab reset.

Current scaffold checkpoints contain Stage 02 sidecars, search spaces, available Stage 01 input artifacts, and optionally the Stage 02 run folder after `run_stage` returns. Formal long-running HPO still needs runner-level incremental trial checkpoints.


In [ ]:
def get_drive_service_for_checkpoint():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError(
            "RUN_STAGE02_CHECKPOINT=True only works inside Colab with Google API dependencies."
        ) from exc
    auth.authenticate_user()
    return build("drive", "v3")

def list_drive_children(service, parent_id, name, mime_type=None):
    escaped_name = quote_drive_query_value(name)
    query_parts = [f"name = '{escaped_name}'", f"'{parent_id}' in parents", "trashed = false"]
    if mime_type:
        query_parts.append(f"mimeType = '{mime_type}'")
    response = service.files().list(
        q=" and ".join(query_parts),
        spaces="drive",
        fields="files(id, name, mimeType, size)",
        pageSize=20,
    ).execute()
    return response.get("files", [])

def ensure_drive_folder(service, parent_id, name):
    folder_mime = "application/vnd.google-apps.folder"
    matches = list_drive_children(service, parent_id, name, folder_mime)
    if len(matches) > 1:
        raise RuntimeError(
            f"Drive checkpoint path is ambiguous: found {len(matches)} folders named {name!r} under {parent_id}."
        )
    if matches:
        return matches[0]["id"]
    metadata = {"name": name, "mimeType": folder_mime, "parents": [parent_id]}
    created = service.files().create(body=metadata, fields="id, name").execute()
    return created["id"]

def ensure_drive_path(service, path_parts):
    folder_id = "root"
    for folder_name in path_parts:
        folder_id = ensure_drive_folder(service, folder_id, folder_name)
    return folder_id

def upload_checkpoint_archive(service, folder_id, archive_path):
    from googleapiclient.http import MediaFileUpload
    media = MediaFileUpload(str(archive_path), mimetype="application/zip", resumable=True)
    metadata = {"name": archive_path.name, "parents": [folder_id]}
    return service.files().create(
        body=metadata,
        media_body=media,
        fields="id, name, size, webViewLink",
    ).execute()

def copy_existing_file(source_path, destination_path, missing_paths):
    if source_path.exists():
        destination_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, destination_path)
    else:
        missing_paths.append(str(source_path))

def assert_stage02_checkpoint_file_scope(output_run_dir):
    if output_run_dir is None:
        return
    output_run_dir = Path(output_run_dir)
    if not output_run_dir.exists():
        return
    forbidden_prefixes = ("03_", "validation_", "test_", "holdout_", "official_validation_")
    forbidden_files = []
    for path in output_run_dir.rglob("*"):
        if path.is_file() and path.name.lower().startswith(forbidden_prefixes):
            forbidden_files.append(str(path))
    if forbidden_files:
        raise ValueError(
            "Stage 02 checkpoint refused possible official validation/test/holdout files: "
            + json.dumps(forbidden_files, indent=2)
        )

def load_stage02_run_safety(output_run_dir):
    safety = {
        "holdout_test_contact": False,
        "official_validation_for_selection": False,
        "ready_for_stage03": False,
        "source_run_manifest": None,
        "source_stage03_handoff": None,
    }
    if output_run_dir is None:
        return safety
    output_run_dir = Path(output_run_dir)
    if not output_run_dir.exists():
        return safety
    run_manifest_path = output_run_dir / "run_manifest.json"
    if run_manifest_path.exists():
        run_manifest = json.loads(run_manifest_path.read_text(encoding="utf-8"))
        safety["source_run_manifest"] = str(run_manifest_path)
        run_manifest_holdout_contact = run_manifest.get("holdout_test_contact")
        if run_manifest_holdout_contact is not False:
            raise ValueError(f"Stage 02 checkpoint requires run_manifest holdout_test_contact=false: {run_manifest_path}")
        safety["holdout_test_contact"] = run_manifest_holdout_contact
        run_manifest_official_validation_for_selection = run_manifest.get("official_validation_for_selection", False)
        if run_manifest_official_validation_for_selection is not False:
            raise ValueError(
                "Stage 02 checkpoint requires run_manifest official_validation_for_selection=false: "
                f"{run_manifest_path}"
            )
        safety["official_validation_for_selection"] = run_manifest_official_validation_for_selection
    stage03_handoff_path = output_run_dir / "02_stage03_handoff.json"
    if stage03_handoff_path.exists():
        stage03_handoff = json.loads(stage03_handoff_path.read_text(encoding="utf-8"))
        safety["source_stage03_handoff"] = str(stage03_handoff_path)
        if stage03_handoff.get("holdout_test_contact") is not False:
            raise ValueError(f"Stage 02 checkpoint requires handoff holdout_test_contact=false: {stage03_handoff_path}")
        safety["ready_for_stage03"] = bool(stage03_handoff.get("ready_for_stage03", False))
    return safety

def build_stage02_checkpoint_archive(label, output_run_dir=None):
    checkpoint_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    checkpoint_dir = CHECKPOINT_ROOT / f"{checkpoint_timestamp}_{label}"
    checkpoint_dir.mkdir(parents=True, exist_ok=False)
    missing_paths = []
    checkpoint_stage01_artifacts = stage02_config["inputs"]["required_stage01_artifacts"]
    assert_stage02_checkpoint_file_scope(output_run_dir)
    run_safety = load_stage02_run_safety(output_run_dir)
    sidecar_dir = checkpoint_dir / "sidecars"
    for source_path in [STAGE_CONFIG_PATH, PROTOCOL_PATH, NOTEBOOK_PATH, STAGE_ENTRYPOINT_PATH]:
        copy_existing_file(source_path, sidecar_dir / source_path.name, missing_paths)
    for family in CORE_HPO_FAMILIES:
        source_path = PROJECT_ROOT / stage02_config["hpo_families"][family]["search_space"]
        copy_existing_file(source_path, checkpoint_dir / "search_spaces" / family / "search_space.yaml", missing_paths)
    stage01_dir = checkpoint_dir / "stage01_inputs"
    for artifact_name in checkpoint_stage01_artifacts:
        copy_existing_file(STAGE01_OUTPUT_DIR / artifact_name, stage01_dir / artifact_name, missing_paths)
    if output_run_dir is not None:
        output_run_dir = Path(output_run_dir)
        if output_run_dir.exists():
            shutil.copytree(output_run_dir, checkpoint_dir / "stage02_run", dirs_exist_ok=True)
        else:
            missing_paths.append(str(output_run_dir))
    manifest = {
        "route": ROUTE,
        "stage_name": STAGE_NAME,
        "checkpoint_label": label,
        "checkpoint_timestamp_utc": checkpoint_timestamp,
        "source_stage01_run_id": STAGE01_RUN_ID,
        "checkpoint_drive_path_parts": CHECKPOINT_DRIVE_PATH_PARTS,
        "included_stage02_sidecars": ["stage config", "protocol", "notebook", "stage entrypoint"],
        "included_stage01_artifacts": [
            name for name in checkpoint_stage01_artifacts if (STAGE01_OUTPUT_DIR / name).exists()
        ],
        "included_output_run_dir": str(output_run_dir) if output_run_dir is not None else None,
        "source_run_manifest": run_safety["source_run_manifest"],
        "source_stage03_handoff": run_safety["source_stage03_handoff"],
        "missing_paths": missing_paths,
        "holdout_test_contact": run_safety["holdout_test_contact"],
        "official_validation_for_selection": run_safety["official_validation_for_selection"],
        "ready_for_stage03": run_safety["ready_for_stage03"],
    }
    manifest_path = checkpoint_dir / "stage02_checkpoint_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    archive_path = Path(shutil.make_archive(str(checkpoint_dir), "zip", root_dir=checkpoint_dir))
    return archive_path, manifest

def write_stage02_drive_checkpoint(label, output_run_dir=None):
    archive_path, manifest = build_stage02_checkpoint_archive(label, output_run_dir=output_run_dir)
    service = get_drive_service_for_checkpoint()
    folder_id = ensure_drive_path(service, CHECKPOINT_DRIVE_PATH_PARTS)
    uploaded = upload_checkpoint_archive(service, folder_id, archive_path)
    print("Stage 02 checkpoint archive:", archive_path)
    print(json.dumps({"checkpoint_manifest": manifest, "drive_upload": uploaded}, indent=2))
    return archive_path, uploaded

if RUN_STAGE02_CHECKPOINT:
    write_stage02_drive_checkpoint("pre_stage02")
else:
    print("RUN_STAGE02_CHECKPOINT=False; no Stage 02 checkpoint archive uploaded.")


## Run Stage 02

The package-backed stage entry point is expected at `lst_models.stages.model_hpo_train_inner.run_stage`. The committed notebook remains inert by default.


In [ ]:
if RUN_STAGE02:
    try:
        from lst_models.stages.model_hpo_train_inner import run_stage
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Missing Stage 02 package entry point: src/lst_models/stages/model_hpo_train_inner.py."
        ) from exc
    result = run_stage(stage02_config)
    display(result)
    if RUN_STAGE02_CHECKPOINT:
        write_stage02_drive_checkpoint("post_stage02", output_run_dir=Path(result.output_dir))
else:
    print("RUN_STAGE02=False; Stage 02 train-inner HPO not executed.")


## Stage 02 Drive Result Backup

After `run_stage` returns, this cell saves the Stage 02 run folder to the canonical durable results path in Google Drive:

```text
My Drive/lst_models/results/02_model_hpo_train_inner/<stage02_run_id>/
```

This is separate from optional checkpoint archives. Downstream stages should read result artifacts from this results path, not from `/content/...` runtime paths and not from checkpoint zips.


In [ ]:
def get_drive_service_for_stage02_result_backup():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError(
            "RUN_STAGE02_DRIVE_BACKUP=True only works inside Colab with Google API dependencies."
        ) from exc
    auth.authenticate_user()
    return build("drive", "v3")

def find_stage02_result_drive_child(service, parent_id, name, mime_type=None):
    escaped_name = quote_drive_query_value(name)
    query_parts = [f"name = '{escaped_name}'", f"'{parent_id}' in parents", "trashed = false"]
    if mime_type:
        query_parts.append(f"mimeType = '{mime_type}'")
    response = service.files().list(
        q=" and ".join(query_parts),
        fields="files(id, name, mimeType, size, webViewLink)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
        pageSize=10,
    ).execute()
    return response.get("files", [])

def ensure_stage02_result_drive_folder(service, parent_id, name):
    folder_mime = "application/vnd.google-apps.folder"
    matches = find_stage02_result_drive_child(service, parent_id, name, folder_mime)
    if len(matches) == 1:
        return matches[0]["id"]
    if len(matches) > 1:
        raise RuntimeError(f"Duplicate Drive folders named {name!r} under parent {parent_id}")
    created = service.files().create(
        body={"name": name, "mimeType": folder_mime, "parents": [parent_id]},
        fields="id, name, webViewLink",
        supportsAllDrives=True,
    ).execute()
    print("Created Drive folder:", name, created.get("webViewLink"))
    return created["id"]

def ensure_stage02_result_drive_path(service, path_parts):
    folder_id = "root"
    for part in path_parts:
        folder_id = ensure_stage02_result_drive_folder(service, folder_id, part)
    return folder_id

def upload_or_update_stage02_result_file(service, parent_id, local_path):
    from googleapiclient.http import MediaFileUpload

    matches = find_stage02_result_drive_child(service, parent_id, local_path.name)
    media = MediaFileUpload(str(local_path), resumable=True)
    if len(matches) == 0:
        uploaded = service.files().create(
            body={"name": local_path.name, "parents": [parent_id]},
            media_body=media,
            fields="id, name, size, webViewLink",
            supportsAllDrives=True,
        ).execute()
        action = "uploaded"
    elif len(matches) == 1:
        uploaded = service.files().update(
            fileId=matches[0]["id"],
            media_body=media,
            fields="id, name, size, webViewLink",
            supportsAllDrives=True,
        ).execute()
        action = "updated"
    else:
        raise RuntimeError(f"Duplicate Drive files named {local_path.name!r} under parent {parent_id}")
    print(f"{action}: {local_path.name}")
    return uploaded

def assert_stage02_result_backup_scope(run_dir):
    forbidden_prefixes = ("03_", "validation_", "test_", "holdout_", "official_validation_")
    forbidden_files = [
        str(path) for path in run_dir.rglob("*")
        if path.is_file() and path.name.lower().startswith(forbidden_prefixes)
    ]
    if forbidden_files:
        raise ValueError(
            "Stage 02 result backup refused possible official validation/test/holdout files: "
            + json.dumps(forbidden_files, indent=2)
        )

def backup_stage02_results_to_drive(output_run_dir):
    run_dir = Path(output_run_dir)
    if not run_dir.exists():
        raise FileNotFoundError(f"Stage 02 output folder not found: {run_dir}")
    required_stage02_files = [
        stage02_config["outputs"]["manifest"],
        stage02_config["outputs"]["artifact_inventory"],
        stage02_config["outputs"]["summary"],
        stage02_config["outputs"]["hpo_plan_ledger"],
        stage02_config["outputs"]["best_params_by_family"],
        stage02_config["outputs"]["stage03_handoff"],
    ]
    missing = [name for name in required_stage02_files if not (run_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing required Stage 02 artifacts before Drive backup: {missing}")
    run_manifest_path = run_dir / stage02_config["outputs"]["manifest"]
    run_manifest = json.loads(run_manifest_path.read_text(encoding="utf-8"))
    if run_manifest.get("holdout_test_contact") is not False:
        raise ValueError(f"Stage 02 result backup requires holdout_test_contact=false: {run_manifest_path}")
    if run_manifest.get("official_validation_for_selection", False) is not False:
        raise ValueError(f"Stage 02 result backup requires official_validation_for_selection=false: {run_manifest_path}")
    stage03_handoff_path = run_dir / stage02_config["outputs"]["stage03_handoff"]
    stage03_handoff = json.loads(stage03_handoff_path.read_text(encoding="utf-8"))
    if stage03_handoff.get("holdout_test_contact") is not False:
        raise ValueError(f"Stage 02 result backup requires handoff holdout_test_contact=false: {stage03_handoff_path}")
    assert_stage02_result_backup_scope(run_dir)
    drive_path_parts = STAGE02_DRIVE_RESULT_PATH_PARTS + [run_dir.name]
    service = get_drive_service_for_stage02_result_backup()
    drive_folder_id = ensure_stage02_result_drive_path(service, drive_path_parts)
    backup_manifest_path = run_dir / "drive_backup_manifest.json"
    local_files = sorted(
        path for path in run_dir.iterdir()
        if path.is_file() and path.name != backup_manifest_path.name
    )
    uploads = [upload_or_update_stage02_result_file(service, drive_folder_id, path) for path in local_files]
    backup_manifest = {
        "stage_name": STAGE_NAME,
        "run_id": run_dir.name,
        "stage02_run_id": run_dir.name,
        "local_output_dir": str(run_dir),
        "stage02_output_dir": str(run_dir),
        "drive_path": "My Drive/" + "/".join(drive_path_parts),
        "drive_path_parts": drive_path_parts,
        "drive_folder_id": drive_folder_id,
        "uploaded_file_names": [upload["name"] for upload in uploads],
        "uploaded_files": uploads,
        "sync_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "holdout_test_contact": run_manifest.get("holdout_test_contact"),
        "official_validation_for_selection": run_manifest.get("official_validation_for_selection"),
        "ready_for_stage03": stage03_handoff.get("ready_for_stage03"),
    }
    backup_manifest_path.write_text(json.dumps(backup_manifest, indent=2), encoding="utf-8")
    manifest_upload = upload_or_update_stage02_result_file(service, drive_folder_id, backup_manifest_path)
    backup_manifest["uploaded_files"].append(manifest_upload)
    backup_manifest["uploaded_file_names"].append(manifest_upload["name"])
    backup_manifest_path.write_text(json.dumps(backup_manifest, indent=2), encoding="utf-8")
    upload_or_update_stage02_result_file(service, drive_folder_id, backup_manifest_path)
    print(json.dumps(backup_manifest, indent=2))
    return backup_manifest

if RUN_STAGE02_DRIVE_BACKUP and RUN_STAGE02:
    if "result" not in globals():
        raise RuntimeError("RUN_STAGE02_DRIVE_BACKUP=True requires running Stage 02 first.")
    stage02_drive_backup_manifest = backup_stage02_results_to_drive(result.output_dir)
else:
    print("RUN_STAGE02_DRIVE_BACKUP is disabled or RUN_STAGE02=False; no Stage 02 result backup uploaded.")


## Artifact Display

After an approved run, display only Stage 02 artifacts from `result.output_dir`. Do not scan for a latest run.


In [ ]:
if RUN_ARTIFACT_DISPLAY:
    import pandas as pd

    if "result" not in globals():
        raise RuntimeError("RUN_ARTIFACT_DISPLAY=True requires running Stage 02 first.")
    run_dir = Path(result.output_dir)
    inventory_path = run_dir / "artifact_inventory.csv"
    summary_path = run_dir / "02_model_hpo_train_inner_summary.csv"
    ledger_path = run_dir / "02_hpo_plan_ledger.csv"
    best_params_path = run_dir / "02_best_params_by_family.json"
    handoff_path = run_dir / "02_stage03_handoff.json"
    for path in [inventory_path, summary_path, ledger_path, best_params_path, handoff_path]:
        require_path(path)
    display(pd.read_csv(inventory_path))
    display(pd.read_csv(summary_path))
    display(pd.read_csv(ledger_path).head(20))
    with best_params_path.open("r", encoding="utf-8") as handle:
        best_params = json.load(handle)
    with handoff_path.open("r", encoding="utf-8") as handle:
        stage03_handoff = json.load(handle)
    assert best_params["holdout_test_contact"] is False
    assert stage03_handoff["holdout_test_contact"] is False
    print(json.dumps(best_params, indent=2))
    print(json.dumps(stage03_handoff, indent=2))
else:
    print("RUN_ARTIFACT_DISPLAY=False; no Stage 02 artifacts displayed.")


## Interpretation Guard

Allowed wording for the current scaffold:

- Stage 02 HPO planning scaffold completed
- Stage 02 blocked because Stage 01 produced no candidate inputs
- Stage 02 did not fit HPO trials and is not ready for Stage 03

Allowed wording only after a future formal HPO implementation writes formal frozen-candidate artifacts:

- frozen train-inner HPO parameters for Stage 03 validation readout
- Stage 02 train-inner HPO completed for approved families
- Stage 02 primary and fallback candidates frozen from train-inner HPO

Do not claim a final model, official validation winner, holdout winner, or test winner from this notebook.
